# Data Cleaning & Preprocessing
## Missing Values, Outliers, and Inconsistencies

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load datasets
clients = pd.read_csv('../data/client_data (1).csv')
price_data = pd.read_csv('../data/price_data (1) (1).csv')

print(f"Clients shape: {clients.shape}")
print(f"Price data shape: {price_data.shape}")

Clients shape: (14606, 26)
Price data shape: (193002, 8)


## 1. Missing Values Analysis

In [2]:
# Analyze missing values in clients data
missing_clients = clients.isnull().sum()
missing_pct = (missing_clients / len(clients)) * 100

missing_df = pd.DataFrame({
    'Column': missing_clients.index,
    'Missing Count': missing_clients.values,
    'Missing %': missing_pct.values
}).sort_values('Missing %', ascending=False)

print("\n=== MISSING VALUES IN CLIENTS DATA ===")
print(missing_df[missing_df['Missing %'] > 0])

# Analyze missing values in price data
missing_price = price_data.isnull().sum()
print("\n=== MISSING VALUES IN PRICE DATA ===")
print(missing_price[missing_price > 0])


=== MISSING VALUES IN CLIENTS DATA ===
Empty DataFrame
Columns: [Column, Missing Count, Missing %]
Index: []

=== MISSING VALUES IN PRICE DATA ===
Series([], dtype: int64)


In [3]:
# Handle missing values in clients data
clients_clean = clients.copy()

# For 'channel_sales' and 'origin_up' (categorical), fill with 'Unknown'
categorical_cols = clients_clean.select_dtypes(include='object').columns
for col in categorical_cols:
    if col != 'id':
        clients_clean[col].fillna('Unknown', inplace=True)

# For numerical columns, fill with median (robust to outliers)
numerical_cols = clients_clean.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_cols:
    if col != 'churn':
        median_val = clients_clean[col].median()
        clients_clean[col].fillna(median_val, inplace=True)

print("\nMissing values after handling:")
print(clients_clean.isnull().sum().sum())


Missing values after handling:
0


/tmp/ipykernel_239691/1363170891.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = clients_clean.select_dtypes(include='object').columns
/tmp/ipykernel_239691/1363170891.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)'

## 2. Outliers Detection (Using Percentiles)

In [4]:
# Define outlier detection using percentiles
def detect_outliers_percentile(data, col, lower_pct=1, upper_pct=99):
    """
    Detect outliers using percentile method.
    Points below lower_pct or above upper_pct are flagged as outliers.
    """
    lower_bound = data[col].quantile(lower_pct / 100)
    upper_bound = data[col].quantile(upper_pct / 100)
    
    outliers = (data[col] < lower_bound) | (data[col] > upper_bound)
    return outliers, lower_bound, upper_bound

# Detect outliers for key numerical columns
key_numerical = ['cons_12m', 'cons_gas_12m', 'net_margin', 'pow_max', 
                 'margin_gross_pow_ele', 'margin_net_pow_ele']

outlier_report = {}
for col in key_numerical:
    if col in clients_clean.columns:
        outliers, lower, upper = detect_outliers_percentile(clients_clean, col)
        outlier_count = outliers.sum()
        outlier_pct = (outlier_count / len(clients_clean)) * 100
        
        outlier_report[col] = {
            'count': outlier_count,
            'percentage': outlier_pct,
            'lower_bound': lower,
            'upper_bound': upper
        }

print("\n=== OUTLIERS DETECTED (1st-99th PERCENTILE) ===")
outlier_df = pd.DataFrame(outlier_report).T
print(outlier_df)
print(f"\nTotal records: {len(clients_clean)}")


=== OUTLIERS DETECTED (1st-99th PERCENTILE) ===
                      count  percentage  lower_bound   upper_bound
cons_12m              273.0    1.869095        19.00  3.329244e+06
cons_gas_12m          143.0    0.979050         0.00  8.889940e+05
net_margin            147.0    1.006436         0.00  9.005955e+02
pow_max               199.0    1.362454        10.35  7.798550e+01
margin_gross_pow_ele  146.0    0.999589         0.00  1.082500e+02
margin_net_pow_ele    146.0    0.999589         0.00  1.082500e+02

Total records: 14606


## 3. Handle Outliers (Cap at Percentiles)

In [ ]:
# Cap outliers at percentiles instead of removing them
clients_clean_capped = clients_clean.copy()

for col in key_numerical:
    if col in clients_clean_capped.columns:
        lower_bound = clients_clean_capped[col].quantile(0.01)
        upper_bound = clients_clean_capped[col].quantile(0.99)
        
        clients_clean_capped[col] = clients_clean_capped[col].clip(lower=lower_bound, upper=upper_bound)

print("Outliers capped at 1st and 99th percentiles.")
print(f"No rows removed - only values clipped.")

## 4. Check for Inconsistencies

In [7]:
# Check logical inconsistencies
inconsistencies = []

# 1. Contract duration should be positive
clients_clean['contract_days'] = (pd.to_datetime(clients_clean['date_end']) - 
                                          pd.to_datetime(clients_clean['date_activ'])).dt.days
negative_contracts = (clients_clean['contract_days'] < 0).sum()
if negative_contracts > 0:
    inconsistencies.append(f"Negative contract duration: {negative_contracts} records")

# 2. Consumption should not be negative
negative_cons = (clients_clean['cons_12m'] < 0).sum() + (clients_clean['cons_gas_12m'] < 0).sum()
if negative_cons > 0:
    inconsistencies.append(f"Negative consumption: {negative_cons} records")

# 3. Margins should not be negative (profit should be non-negative by definition)
negative_margins = (clients_clean['net_margin'] < 0).sum()
if negative_margins > 0:
    inconsistencies.append(f"Negative margins: {negative_margins} records")


print("\n=== INCONSISTENCIES FOUND ===")
if inconsistencies:
    for issue in inconsistencies:
        print(f"  ⚠️  {issue}")
else:
    print("  ✓ No major inconsistencies detected")


=== INCONSISTENCIES FOUND ===
  ✓ No major inconsistencies detected


## 6. Save Cleaned Data

In [ ]:
# Save cleaned datasets
clients_clean_capped.to_csv('../data/clients_cleaned.csv', index=False)
price_clean.to_csv('../data/price_data_cleaned.csv', index=False)

print("\n✓ Cleaned data saved:")
print("  - ../data/clients_cleaned.csv")
print("  - ../data/price_data_cleaned.csv")